# 05 Dashboard Export

- prepara un export leggero partendo dal primo mart dichiarato in config
- non scrive file finche `EXPORT = False`
- usa `_tmp/` per evitare output committati nel repo

In [ ]:
from pathlib import Path
import duckdb
import yaml

ROOT = Path('.').resolve()
DATASET_YML = (ROOT / 'dataset.yml').resolve() if (ROOT / 'dataset.yml').exists() else (ROOT / '..' / 'dataset.yml').resolve()
CFG = yaml.safe_load(DATASET_YML.read_text(encoding='utf-8'))
BASE_DIR = DATASET_YML.parent
OUT_ROOT = (BASE_DIR / CFG.get('root', '.')).resolve()
DATASET = CFG['dataset']['name']
YEARS = CFG['dataset']['years']
YEAR_INDEX = 0
YEAR = YEARS[YEAR_INDEX] if YEARS and 0 <= YEAR_INDEX < len(YEARS) else YEARS[0]
TABLES = CFG.get('mart', {}).get('tables', [])
TABLE_INDEX = 0
SELECTED_TABLE = TABLES[TABLE_INDEX] if TABLES and 0 <= TABLE_INDEX < len(TABLES) else (TABLES[0] if TABLES else {'name': 'mart_ok'})
TABLE_NAME = SELECTED_TABLE['name']
MART_PATH = OUT_ROOT / 'data' / 'mart' / DATASET / str(YEAR) / f'{TABLE_NAME}.parquet'
OUT_DIR = (BASE_DIR / '_tmp').resolve()
EXPORT = False
{'YEARS': YEARS, 'YEAR_INDEX': YEAR_INDEX, 'TABLES': [table['name'] for table in TABLES], 'TABLE_INDEX': TABLE_INDEX, 'TABLE_NAME': TABLE_NAME, 'MART_PATH': str(MART_PATH)}

In [ ]:
con = duckdb.connect()
YEAR_COL = None
METRIC_COL = None
export_df = None

def choose_columns(path):
    rows = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{path.as_posix()}')").fetchall()
    year_col = next((row[0] for row in rows if str(row[0]).lower() == 'year' or 'anno' in str(row[0]).lower()), None)
    numeric_rows = [row[0] for row in rows if any(token in str(row[1]).upper() for token in ['INT', 'DECIMAL', 'DOUBLE', 'FLOAT', 'REAL', 'BIGINT'])]
    metric_col = next((col for col in numeric_rows if any(token in col.lower() for token in ['value', 'tot', 'importo', 'ammontare', 'saldo', 'spese', 'entrate', 'pct', 'percent'])), None)
    if metric_col is None and numeric_rows:
        metric_col = numeric_rows[0]
    return year_col, metric_col

if MART_PATH.exists():
    YEAR_COL, METRIC_COL = choose_columns(MART_PATH)
    print({'YEAR_COL': YEAR_COL, 'METRIC_COL': METRIC_COL})
else:
    print('MART parquet not found.')

In [ ]:
if MART_PATH.exists() and YEAR_COL and METRIC_COL:
    export_df = con.execute(
        f"SELECT {YEAR_COL} AS year_like, {METRIC_COL} AS metric_value FROM read_parquet('{MART_PATH.as_posix()}') ORDER BY 1"
    ).df()
elif MART_PATH.exists():
    export_df = con.execute(f"SELECT * FROM read_parquet('{MART_PATH.as_posix()}') LIMIT 1000").df()
else:
    export_df = None

if export_df is not None:
    display(export_df.head())

In [ ]:
if EXPORT and export_df is not None:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    csv_path = OUT_DIR / f'{TABLE_NAME}_dashboard.csv'
    parquet_path = OUT_DIR / f'{TABLE_NAME}_dashboard.parquet'
    export_df.to_csv(csv_path, index=False)
    con.register('export_df_view', export_df)
    con.execute(f"COPY (SELECT * FROM export_df_view) TO '{parquet_path.as_posix()}' (FORMAT PARQUET)")
    print(csv_path)
    print(parquet_path)
else:
    print('Export disabled. Set EXPORT = True to write files into ../_tmp/.')